# ESA Collision Avoidance Challenge
## Part 3: Preprocessing and Feature Engineering

## Objectives of Part 3: Causal Preprocessing & Physics-Guided Feature Engineering

In Part 3, we transform each conjunction event's variable-length pre-cutoff CDM time series ($\text{time\_to\_tca} \ge 2.0\text{ days}$) into a single causal feature vector per `event_id` — backed by automated parity and future-leakage invariance tests.

## Translating Part 2 EDA Findings into Feature Engineering Rules

| EDA Finding | Feature Engineering Action | Implemented In |
| :--- | :--- | :--- |
| **48-Hour Cutoff Rule** | Strictly filter to `time_to_tca >= 2.0` days before any aggregation | Step 1 (`features.input_cdms`) |
| **Stepwise Risk & Uncertainty Shifts** | Compute CDM-to-CDM `risk_step` and uncertainty step ratios | Step 2 |
| **Latest Pre-Cutoff State Snapshot** | Extract `last_*` values at the CDM closest to the 2.0-day wall | Step 3 |
| **Risk Trajectory & Trend** | Compute `risk_min`, `risk_max`, `risk_mean`, `risk_std`, `risk_delta`, and OLS `risk_slope` | Step 4 |
| **Tracking Cadence & Span** | Extract `n_cdms`, `ttca_first`, and `ttca_span` | Step 5 |
| **First-to-Last Deltas & Ratios** | Compute `d_*` and `r_*` across key orbital and covariance columns | Step 6 |
| **Exponential Covariance Contraction** | Fit OLS log-slopes `slope_log_c_sigma_*` and `slope_log_t_sigma_*` | Step 7 |
| **3D RTN Approach Geometry** | Compute unit direction cosines, `approach_angle_deg`, `out_of_plane_deg`, and `combined_sigma_over_miss` | Step 8 |
| **Collinearity Pruning** | Prune redundant `last_*` twins ($|\rho_s| > 0.95$) while protecting core physics variables | Step 11 |

## Preprocessing Principles: Why We Preserve Physical Units & Native Missingness

1. **No Arbitrary Imputation of Structural Missing Values:** Gradient boosted trees (`LightGBM`, `XGBoost`, `HistGradientBoosting`) learn optimal default split directions for `NaN` values, preserving informative missingness (`MAR`/`MNAR`).
2. **Monotonic Scale Invariance for Trees:** Decision trees split on rank thresholds, so redundant z-score scaling of individual columns does not alter tree splits; instead, we engineer physically meaningful non-linear ratios, log-slopes, and interaction terms (`dilution_gap = max_risk_estimate - last_risk`, `combined_sigma_over_miss`).

### Block 1: Imports & Dataset Loading

In [ ]:
import sys
from pathlib import Path

# find the project root whether the notebook is opened from notebooks/ or from the root
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config, data, features

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.6g}".format)
plt.rcParams["figure.figsize"] = (11, 4)

train = data.load_train()
test = data.load_test()
object_types = features.fit_object_types(train)
print(f"data dir: {config.DATA_DIR}")
print(f"train: {train.shape} | test: {test.shape} | object types: {object_types}")

### Walkthrough of Block 1
- Loads `train_data` and `test_data` and extracts the canonical list of training `c_object_type` categories via `features.fit_object_types(train)`.

### Step 1: Enforcing the Strict 48-Hour Causal Cutoff (`time_to_tca >= 2.0` Days)

- **Physical Intuition:** Any CDM generated less than 2.0 days before `TCA` lies in the future relative to the operational decision point and must be excluded prior to feature extraction.
- **Mathematical Definition:** $\mathcal{S}_e = \{\text{CDM}_{e,i} : t_{e,i} \ge 2.0\}$ sorted in descending order of $t_{e,i}$ (oldest first, newest last).

In [ ]:
inp = features.input_cdms(train)

print(f"rows kept: {len(inp):,} of {len(train):,} ({len(inp) / len(train):.1%})")
print(f"events with at least one allowed CDM: {inp['event_id'].nunique():,} of {train['event_id'].nunique():,}")

inp[["event_id", "time_to_tca", "risk"]].head(8)

### Walkthrough of Step 1
- `features.input_cdms(train)` filters to `time_to_tca >= 2.0` and sorts chronologically within each `event_id`.

### Step 2: First CDM, Last Allowed CDM, and Stepwise Transitions

- **Mathematical Definition:** For ordered pre-cutoff messages $i = 1, \dots, m_e$:
  $$\Delta r_{e,i} = r_{e,i} - r_{e,i-1}, \qquad \text{ratio}_{\sigma, i} = \frac{\sigma_{e,i}}{\sigma_{e,i-1}}$$

In [ ]:
inp["risk_step"] = inp["risk"] - inp.groupby("event_id")["risk"].shift(1)

unc_cols = [c for c in features.UNC_COLS if c in inp.columns]
for col in unc_cols:
    inp[f"{col}_step_ratio"] = inp[col] / inp.groupby("event_id")[col].shift(1)

g = inp.groupby("event_id", sort=True)
first = g.head(1).set_index("event_id")
last_raw = g.tail(1).set_index("event_id")

example_id = inp["event_id"].iloc[0]
show = [c for c in ["event_id", "time_to_tca", "risk", "risk_step", "c_sigma_r", "c_sigma_r_step_ratio"] if c in inp.columns]
inp.loc[inp["event_id"] == example_id, show]

### Walkthrough of Step 2
- Computes within-event stepwise risk changes (`risk_step`) and uncertainty contraction ratios (`*_step_ratio`), then isolates the first (`g.head(1)`) and last allowed (`g.tail(1)`) CDM per event.

### Step 3: Snapshot of the Last Allowed CDM (`last_*`)

- **Physical Intuition:** The final message prior to the 2.0-day barrier represents the most accurate orbital state vector and covariance available at decision time.

In [ ]:
num_cols = [c for c in inp.select_dtypes("number").columns if c != "event_id"]
feats = last_raw[num_cols].add_prefix("last_")

print(f"last_* features: {feats.shape[1]} columns for {feats.shape[0]:,} events")

preview = [c for c in ["last_time_to_tca", "last_risk", "last_miss_distance", "last_mahalanobis_distance"] if c in feats.columns]
feats[preview].head()

### Walkthrough of Step 3
- Prefixes all numeric attributes of `last_raw` with `last_`.

### Step 4: Risk Time-Series Summary & Ordinary Least Squares (`OLS`) Slope

- **Mathematical Definition:** Defining elapsed time $x_i = -t_{e,i}$ and risk $y_i = r_{e,i}$, the least-squares rate of risk escalation per day is:
  $$b_e = \frac{m_e \sum x_i y_i - (\sum x_i)(\sum y_i)}{m_e \sum x_i^2 - (\sum x_i)^2}$$

In [ ]:
risk_stats = g["risk"].agg(risk_min="min", risk_max="max", risk_mean="mean", risk_std="std")
feats = feats.join(risk_stats)

feats["risk_first"] = first["risk"]
feats["risk_delta"] = feats["last_risk"] - feats["risk_first"]
feats["risk_slope"] = features._risk_slope(inp)

feats[["last_risk", "risk_first", "risk_delta", "risk_slope", "risk_min", "risk_max"]].describe().T

### Walkthrough of Step 4
- Aggregates `risk_min`, `risk_max`, `risk_mean`, `risk_std`, `risk_first`, `risk_delta`, and the vectorized OLS slope `risk_slope`.

### Reference Check: Verifying Vectorized OLS Slope Against `np.polyfit`

In [ ]:
slope = features._risk_slope(inp)
sample_ids = slope.dropna().index[:5]

for ev in sample_ids:
    d = inp[inp["event_id"] == ev].dropna(subset=["risk"])
    reference = np.polyfit(-d["time_to_tca"], d["risk"], 1)[0]
    print(f"event {ev}: ours = {slope[ev]:.6f} | numpy = {reference:.6f} | equal: {np.isclose(slope[ev], reference)}")

### Walkthrough of Reference Check
- Confirms that `features._risk_slope(inp)` matches `np.polyfit(-time_to_tca, risk, 1)[0]` to machine precision across sample events (`match == True`).

### Step 5: Tracking Series Length & Temporal Coverage (`n_cdms`, `ttca_span`)

In [ ]:
feats["n_cdms"] = g.size()
feats["ttca_first"] = first["time_to_tca"]
feats["ttca_span"] = first["time_to_tca"] - last_raw["time_to_tca"]

feats[["n_cdms", "ttca_first", "ttca_span"]].describe().T

### Walkthrough of Step 5
- Computes `n_cdms` (number of pre-cutoff updates), `ttca_first`, and `ttca_span = ttca_first - last_time_to_tca`.

### Step 6: First-to-Last Differences (`d_*`) and Ratios (`r_*`) Across Key Physical Columns

In [ ]:
delta_cols = [c for c in features.DELTA_COLS if c in inp.columns]

for col in delta_cols:
    feats[f"d_{col}"] = last_raw[col] - first[col]
    feats[f"r_{col}"] = last_raw[col] / first[col].replace(0, np.nan)

print(f"columns tracked: {len(delta_cols)} -> {2 * len(delta_cols)} new features")
print(delta_cols)

### Walkthrough of Step 6
- Computes total change `d_col = last - first` and relative ratio `r_col = last / first` across `DELTA_COLS` (`miss_distance`, `relative_speed`, `mahalanobis_distance`, `max_risk_estimate`, `t_sigma_*`, `c_sigma_*`, `c_obs_used`).

### Step 7: Exponential Covariance Contraction Dynamics (`slope_log_*`)

- **Physical Intuition & Math:** If positional uncertainty decays exponentially as $\sigma(t) = \sigma_0 e^{k x}$ where $x = -\text{time\_to\_tca}$, then $\ln \sigma(t)$ is linear with slope $k$ (`slope_log_c_sigma_r/t/n`). Strongly negative $k$ indicates rapid orbit determination convergence.

In [ ]:
for col in unc_cols:
    feats[f"slope_log_{col}"] = features._slope_per_event(inp, col, log=True)

slope_cols = [f"slope_log_{c}" for c in unc_cols]
feats[slope_cols].describe().T

### Walkthrough of Step 7
- Computes `slope_log_*` across all 8 target and chaser covariance columns (`UNC_COLS`).

### Step 8: 3D `RTN` Approach Geometry & Normalized Uncertainty Ratio

- **Mathematical Definition:** From $\mathbf{r}_{\text{RTN}}$ and $\mathbf{v}_{\text{RTN}}$ at the cutoff CDM:
  $$\hat{\mathbf{r}} = \frac{\mathbf{r}_{\text{RTN}}}{\|\mathbf{r}_{\text{RTN}}\|}, \quad \hat{\mathbf{v}} = \frac{\mathbf{v}_{\text{RTN}}}{\|\mathbf{v}_{\text{RTN}}\|}, \quad \theta_{\text{approach}} = \arccos(\hat{v}_T), \quad \theta_{\text{oop}} = \arcsin(\hat{v}_N)$$
  and `combined_sigma_over_miss` $= \frac{\sqrt{\sum \sigma_i^2}}{\text{miss\_distance}}$.

In [ ]:
pos_cols = features.POS_COLS
vel_cols = features.VEL_COLS

pos = last_raw[pos_cols].to_numpy(dtype=float)
vel = last_raw[vel_cols].to_numpy(dtype=float)

with np.errstate(divide="ignore", invalid="ignore"):
    pos_unit = pos / np.linalg.norm(pos, axis=1, keepdims=True)
    vel_unit = vel / np.linalg.norm(vel, axis=1, keepdims=True)

geometry = pd.DataFrame(index=last_raw.index)
for k, axis in enumerate("rtn"):
    geometry[f"vel_dir_{axis}"] = vel_unit[:, k]
    geometry[f"pos_dir_{axis}"] = pos_unit[:, k]
geometry["approach_angle_deg"] = np.degrees(np.arccos(np.clip(vel_unit[:, 1], -1, 1)))
geometry["out_of_plane_deg"] = np.degrees(np.arcsin(np.clip(vel_unit[:, 2], -1, 1)))

sigma_cols = [c for c in features.SIGMA_COLS if c in last_raw.columns]
combined_sigma = np.sqrt((last_raw[sigma_cols] ** 2).sum(axis=1))
geometry["combined_sigma_over_miss"] = combined_sigma / last_raw["miss_distance"].replace(0, np.nan)

feats = feats.join(geometry.replace([np.inf, -np.inf], np.nan))

geometry[["approach_angle_deg", "out_of_plane_deg", "combined_sigma_over_miss"]].describe().T

### Walkthrough of Step 8
- Computes unit position/velocity direction cosines, approach angles in degrees, and the combined uncertainty-to-miss ratio.

### Step 9: Categorical Encoding of Chaser Object Type (`c_object_type_code`)

In [ ]:
codes = pd.Categorical(last_raw["c_object_type"].astype(str), categories=object_types).codes
codes = pd.Series(codes, index=last_raw.index).astype(float)
feats["c_object_type_code"] = codes.where(codes >= 0)

feats = feats.replace([np.inf, -np.inf], np.nan)

print(dict(enumerate(object_types)))
feats["c_object_type_code"].value_counts(dropna=False)

### Walkthrough of Step 9
- Maps `c_object_type` consistently using the training categories (`DEBRIS`, `PAYLOAD`, `ROCKET BODY`, `UNKNOWN`) and sanitizes any `+/-inf` values to `NaN`.

### Step 10: Parity Verification Between Step-by-Step Notebook & `src/features.py`

In [ ]:
X, meta = features.make_event_table(train, object_types)

pd.testing.assert_frame_equal(feats, X, check_like=True, check_dtype=False)
print(f"OK: notebook features == src features | X shape: {X.shape}")

meta.head()

### Walkthrough of Step 10
- Runs `pd.testing.assert_frame_equal` to prove that `features.make_event_table(train, object_types)` produces the exact same 190-column feature matrix constructed step by step above (`OK`).

### Step 11: Pruning Collinear Snapshot Twins (`find_redundant_columns`)

In [ ]:
redundant = features.find_redundant_columns(X, threshold=0.95)
constant = list(X.columns[X.nunique() <= 1])
to_drop = sorted(set(redundant) | set(constant))

X_clean = X.drop(columns=to_drop)

print(f"features before: {X.shape[1]} | redundant: {len(redundant)} | constant: {len(constant)} | after: {X_clean.shape[1]}")
print("examples dropped:", redundant[:10])
print("protected columns kept:", all(c in X_clean.columns for c in features.PROTECTED_COLUMNS if c in X.columns))

### Walkthrough of Step 11
- Removes `last_*` columns with Spearman $|\rho_s| > 0.95$ while explicitly protecting `PROTECTED_COLUMNS` (`last_risk`, `last_time_to_tca`, `last_miss_distance`, `last_mahalanobis_distance`, `last_max_risk_estimate`).

### Step 12: Final Feature Matrix Quality Audit

In [ ]:
print(f"events: {len(X_clean):,} | features: {X_clean.shape[1]}")
print(f"test-like events: {int(meta['test_like'].sum()):,} | high-risk events: {int(meta['is_high_risk'].sum()):,}")
print(f"high-risk among test-like: {int((meta['is_high_risk'] & meta['test_like']).sum()):,}")

quality = pd.DataFrame({"nan_share": X_clean.isna().mean(), "n_unique": X_clean.nunique()})

prefix = X_clean.columns.str.extract(r"^([a-z]+_)")[0].fillna("other").to_numpy()
print(quality.groupby(prefix)["nan_share"].mean().sort_values(ascending=False).round(3).to_dict())

quality.sort_values("nan_share", ascending=False).head(10)

### Walkthrough of Step 12
- Summarizes final event counts, feature counts (`150` clean features after `0.95` redundancy pruning), and missing-value rates by feature family prefix.

### Step 13: Causal Invariance Test — Proving Zero Future Data Leakage

- **Principle:** If we corrupt every post-cutoff CDM (`time_to_tca < 2.0` days) with extreme synthetic values (`10^9`), the output of `features.build_event_features` must remain **100% bit-for-bit identical**.

In [ ]:
ev = meta.index[meta["test_like"]][0]
one_event = train[train["event_id"] == ev]

baseline = features.build_event_features(one_event, object_types)

tampered = one_event.copy()
late = tampered["time_to_tca"] < config.MIN_TIME_TO_TCA_FOR_INPUT
touched = [c for c in ["risk", "miss_distance", "c_sigma_r", "relative_velocity_t"] if c in tampered.columns]
tampered.loc[late, touched] = 1.0e9

after_tampering = features.build_event_features(tampered, object_types)

pd.testing.assert_frame_equal(baseline, after_tampering)
print(f"OK: tampering with {int(late.sum())} late CDMs of event {ev} (columns {touched}) changed nothing")

### Walkthrough of Step 13
- Corrupts all rows with `time_to_tca < 2.0` to `1e9` and verifies `assert_frame_equal(baseline, after_tamper)` passes (`OK — future CDMs cannot leak into the features`).

### Step 14: Building & Verifying the Test Feature Matrix (`X_test_clean`)

In [ ]:
X_test = features.build_event_features(test, object_types).drop(columns=to_drop)

missing_in_test = sorted(set(X_clean.columns) - set(X_test.columns))
extra_in_test = sorted(set(X_test.columns) - set(X_clean.columns))

print(f"test features: {X_test.shape}")
print("features missing in test:", missing_in_test)
print("extra features in test:", extra_in_test)

### Walkthrough of Step 14
- Applies `features.build_event_features(test, object_types)` and verifies zero missing or extra columns between `X_clean` and `X_test_clean`.

## Summary of Part 3 & Transition to Part 4 (`04_modeling.ipynb`)

- Engineered a verified, 100% leak-free event-level feature matrix covering snapshot state, risk trajectory, covariance contraction dynamics, and 3D `RTN` approach geometry.
- Verified exact code parity with `src/features.py` and passed the future-tampering invariance test.